1.Importing necessary Libraries,Creating plt and loading Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
params = {'legend.fontsize': 'x-large',
          'figure.figsize': (20, 6),
         'axes.labelsize': 'x-large',
         'axes.titlesize':'x-large',
         'xtick.labelsize':'x-large',
         'ytick.labelsize':'x-large',
          'axes.titlesize': 16,
          'axes.labelweight': 'bold'}
plt.rcParams.update(params)

In [ ]:
df=pd.read_csv(r"adverts.csv")
df.head(10)
df = df.sample(frac=0.5, random_state=42)

1.1 

In [ ]:
quantitative_features=df.select_dtypes(include=["int64","float64"])
quantitative_features.head()

In [ ]:
qualitative_features=df.select_dtypes(include=["object"])
qualitative_features.head()

Meaning and type of features

In [ ]:
df.shape

Analysis of features

In [ ]:
sns.histplot(data=df, x='vehicle_condition')

In [ ]:
sns.boxplot(data=df, x='mileage')

1.2 Analysis of predictive power of features

In [ ]:
sns.scatterplot(data=df, x='mileage', y='price')

In [ ]:
sns.scatterplot(data=df, x='year_of_registration', y='price')

In [ ]:
sns.scatterplot(data=df, x='body_type', y='price')

1.3 Data Processing for Data exploration and visualisation

In [ ]:
df.info()

In [ ]:
variables = pd.DataFrame(columns=['Variable','Number of unique values','Number of nulls'])

for i, var in enumerate(df.columns):
    variables.loc[i] = [var, df[var].nunique(), df[var].isnull().sum()]

variables

In [ ]:
df.describe()

In [ ]:
sns.barplot(data=df,x='fuel_type',y='price')

2.1 Dealing with missing values and outliers

In [ ]:
df.loc[df['year_of_registration' ]<=1950]

In [ ]:
df.loc[df['reg_code']=='08','year_of_registration']=2008

In [ ]:
df=df[df["year_of_registration"]>=1950]

In [ ]:
sns.boxplot(data=df, x='year_of_registration')

In [ ]:
df.loc[df["vehicle_condition"]=="NEW","year_of_registration"]=df.loc[df["vehicle_condition"]=="NEW","year_of_registration"].fillna(2020)

In [ ]:
variables = pd.DataFrame(columns=['Variable','Number of unique values','Number of nulls'])

for i, var in enumerate(df.columns):
    variables.loc[i] = [var, df[var].nunique(), df[var].isnull().sum()]

variables

In [ ]:
df.info()

In [ ]:
variables = pd.DataFrame(columns=['Variable','Number of unique values','Number of nulls'])

for i, var in enumerate(df.columns):
    variables.loc[i] = [var, df[var].nunique(), df[var].isnull().sum()]

variables

In [ ]:
sns.boxplot(data=df, x='mileage')

In [ ]:
df

In [ ]:
df=df.dropna()

In [ ]:
#Filling all the NEW cars mileage to 0
df.loc[(df['vehicle_condition'] == 'NEW')&(df['mileage']>0), 'mileage']=0

In [ ]:
import pandas as pd
from scipy.stats import zscore
df['z_score'] = zscore(df['mileage'])
threshold = 3
df= df[abs(df['z_score']) < threshold]
df= df.drop(columns=['z_score'])
print("Filtered DataFrame without outliers:")
print(df)

In [ ]:
sns.boxplot(data=df, x='mileage')

In [ ]:
import datetime
current_year = datetime.datetime.now().year
df['car_age'] = current_year - df['year_of_registration']

In [ ]:
df

In [ ]:
df['log_mileage'] = np.log(df['mileage']+1)

In [ ]:
df['log_price'] = np.log(df['price'])

In [ ]:
df

In [ ]:
standard_colour_count= df['standard_colour'].value_counts()
df['standard_colour'] = df['standard_colour'].apply(lambda x: x if standard_colour_count[x] >= 10000 else 'others')

In [ ]:
df= df.drop(columns=["public_reference","reg_code","price"])

In [ ]:
import sklearn
sklearn.set_config(transform_output="pandas")

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, ParameterGrid

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer

In [ ]:
categorical_transformer=Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
    ]
)

In [ ]:
categorical_transformer

In [ ]:
numeric_transformer=Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scalar",MinMaxScaler())
    ]
)

In [ ]:
numeric_transformer

In [ ]:
cat_feat = ['standard_colour', 'standard_make', 'standard_model','vehicle_condition','fuel_type','crossover_car_and_van']
num_feat = ['year_of_registration', 'car_age', 'log_mileage']

In [ ]:
preprocessor=ColumnTransformer(
    transformers=[
        ("cateo",categorical_transformer,cat_feat),
        ("num",numeric_transformer,num_feat)
    ]
)

In [ ]:
preprocessor

In [ ]:
X = df.drop(columns=['log_price'])
y = df['log_price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
knn_regressor=Pipeline([
    ("preprocessor",preprocessor),
    ("knnregressor",KNeighborsRegressor(n_neighbors=4))
])
knn_regressor.fit(X_train,y_train)

In [ ]:
testscore=knn_regressor.score(X_test, y_test)
trainscore=knn_regressor.score(X_train, y_train)
trainscore

In [ ]:
y_pred_knn = knn_regressor.predict(X_test)

In [ ]:
y_pred_knn

In [ ]:
param_grid = {
    'knnregressor__n_neighbors': [3, 5],  # Correctly reference the parameter in the pipeline
    'knnregressor__weights': ['uniform', 'distance']  # Example of additional parameters to tune
}

In [ ]:
from sklearn.model_selection import ParameterGrid
list(ParameterGrid(param_grid))

In [ ]:
knn_regressor_grid = GridSearchCV(knn_regressor, param_grid, cv=3,return_train_score=True)
knn_regressor_grid.fit(X_train, y_train)

In [ ]:
gs_df_knn = pd.DataFrame(knn_regressor_grid.cv_results_)
print(gs_df_knn)

In [ ]:
from sklearn.tree import DecisionTreeRegressor
Dt_regressor=Pipeline([
    ("preprocessor",preprocessor),
    ("regressor",DecisionTreeRegressor(random_state=42))
])
Dt_regressor.fit(X_train,y_train)

In [ ]:
test_score=Dt_regressor.score(X_test, y_test)
train_score=Dt_regressor.score(X_train, y_train)

In [ ]:
y_pred_Dt= Dt_regressor.predict(X_test)

In [ ]:
y_pred_Dt

In [ ]:
param_grid = {
    'regressor__max_depth': [3, 5, 10, None],
    'regressor__min_samples_split': [2, 5, 10],
    'regressor__min_samples_leaf': [1, 2, 4],
    'regressor__max_features': [None, 'sqrt', 'log2']
}


In [ ]:
list(ParameterGrid(param_grid))

In [ ]:
from sklearn.model_selection import GridSearchCV
grid_search_dt = GridSearchCV(estimator=Dt_regressor, param_grid=param_grid, cv=5, scoring='r2', return_train_score=True)
grid_search_dt.fit(X_train,y_train)
gs_df_dt = pd.DataFrame(grid_search_dt.cv_results_)


In [ ]:
gs_df_dt

In [ ]:
from sklearn.linear_model import LinearRegression
linear_regressor=Pipeline([
    ("preprocessor",preprocessor),
    ("linearregressor",LinearRegression())
])
linear_regressor.fit(X_train,y_train)

In [ ]:
linear_regressor.score(X_test, y_test)

In [ ]:
y_pred_Lr= linear_regressor.predict(X_test)

In [ ]:
y_pred_Lr

In [ ]:
from sklearn.linear_model import Ridge

linear_regressor = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('ridge', Ridge())
])

In [ ]:
param_grid = {
    'ridge__alpha': [0.1, 1, 10, 100],
    'ridge__fit_intercept': [True, False],
    'ridge__solver': ['auto', 'saga', 'lsqr']
}

In [ ]:
grid_search_lr = GridSearchCV(estimator=linear_regressor, param_grid=param_grid, cv=2, scoring='r2', return_train_score=True)
grid_search_lr.fit(X_train, y_train)
gs_df_linear_lr = pd.DataFrame(grid_search_lr.cv_results_)

In [ ]:
gs_df_linear_lr

In [ ]:
knn_regressor_grid.best_params_

In [ ]:
knn_regressor_grid.best_estimator_

4.1

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
y_pred_knn_score =knn_regressor_grid.predict(X_test)

In [ ]:
r2_score_knn=r2_score(y_test,y_pred_knn_score)
mae_score_knn=mean_absolute_error(y_test,y_pred_knn_score)
mse_score_knn= mean_squared_error(y_test,y_pred_knn_score)

In [ ]:
print(r2_score_knn)
print(mae_score_knn)
print(mse_score_knn)

In [ ]:
y_pred_dt_score =grid_search_dt.predict(X_test)

In [ ]:
r2_score_dt=r2_score(y_test,y_pred_dt_score)
mae_score_dt=mean_absolute_error(y_test,y_pred_dt_score)
mse_score_dt= mean_squared_error(y_test,y_pred_dt_score)

In [ ]:
print(r2_score_dt)
print(mae_score_dt)
print(mse_score_dt)

In [ ]:
y_pred_lr=grid_search_lr.predict(X_test)

In [ ]:
r2_lr_score = r2_score(y_test,y_pred_lr)
mae_lr_score = mean_absolute_error(y_test,y_pred_lr)
mse_lr_score = mean_squared_error(y_test,y_pred_lr)

In [ ]:
print(r2_lr_score)
print(mae_lr_score)
print(mse_lr_score)

In [ ]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(knn_regressor, X_train, y_train, cv=3, scoring='r2')
print("Cross-Validation R² Scores:", cv_scores)
print("Mean Cross-Validation R² Score:", np.mean(cv_scores))

In [ ]:
heatmap_data = np.array(cv_scores).reshape(1, -1)  
plt.figure(figsize=(8, 2))
sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", cbar=True, fmt=".2f", xticklabels=[f"Fold {i}" for i in range(1, len(cv_scores) + 1)], 
            yticklabels=["CV Scores"])
plt.title("Cross-Validation R² Scores Heatmap")
plt.show()

In [ ]:
models = ["KNN", "Decision Tree", "Linear Regression"]
scores = [r2_score_knn, r2_score_dt,r2_lr_score] 
heatmap_data = np.array(scores).reshape(1, -1)
plt.figure(figsize=(8, 2))
sns.heatmap(heatmap_data, annot=True, cmap="coolwarm", cbar=True, fmt=".2f",
            xticklabels=models, yticklabels=["R² Scores"])
plt.title("Model Comparison Heatmap")
plt.show()


In [ ]:
feature_importances = Dt_regressor.named_steps['regressor'].feature_importances_
feature_names = num_feat+ list(preprocessor.named_transformers_['cateo']['encoder'].get_feature_names_out(cat_feat))
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

In [ ]:
print(importance_df)

In [ ]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
permutation= permutation_importance(knn_regressor, X_test, y_test, n_repeats=10, random_state=42)
feature_importance= pd.Series(permutation.importances_mean, index=X_test.columns)
print(feature_importance)

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(y_test,y_pred_knn_score, color='blue', alpha=0.6,edgecolors='w')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], linestyle='--', color='red', linewidth=2, label='perfect prediction')
plt.title('actual vs predicted value for knn regressor')
plt.xlabel('actual value')
plt.ylabel('predicted values')
plt.legend()
plt.show()

In [ ]:
residuals = y_test - y_pred_knn_score
plt.hist(residuals, bins=20, edgecolor='k', alpha=0.7)
plt.xlabel("Residuals")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.show()

In [ ]:
error_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred_knn_score,
    'Error': residuals
}).reset_index()

error_df = error_df.reindex(error_df['Error'].abs().sort_values(ascending=False).index)
print(error_df.head(5))